# Railroad Interest → Labor Coverage Regression

Tests whether newspaper owners with railroad financial ties produced more anti-labor coverage.

**Unit of observation:** newspaper × year  
**Outcome:** `anti_labor_intensity` = anti-labor articles / total labor articles  
**Treatment:** `railroad_interest` = 1 if any coded owner had a financial railroad tie (stockholder, board member, company owner, donation, professional services, or family connection)

**Control group:** only persons with at least one explicit 0 across the railroad interest fields and no positive values — "confirmed non-railroad". Persons with all-NaN coding (never coded) are dropped from the analysis.

In [ ]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

DB_PATH   = '../data/newspapers.db'
OE_PATH   = '../data/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/personnel_coding/combined_coding.csv'

## 1. Build railroad interest indicator

In [ ]:
coding = pd.read_csv(CODE_PATH)

RAILROAD_COLS = [
    'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner',
    'railroad_donation', 'railroad_professional_services', 'family_connection_railroad'
]

# railroad_interest=1: any column is > 0
coding['railroad_interest'] = (coding[RAILROAD_COLS].fillna(0) > 0).any(axis=1).astype(int)

# confirmed_control=1: at least one column has an explicit 0, and none are > 0
has_explicit_zero = (coding[RAILROAD_COLS] == 0).any(axis=1)
coding['confirmed_control'] = has_explicit_zero & (coding['railroad_interest'] == 0)

# Keep only persons who are either treated or confirmed controls
person_rr = coding[coding['railroad_interest'] | coding['confirmed_control']].copy()
person_rr = person_rr[['person_id', 'railroad_interest']].dropna(subset=['person_id'])
person_rr['person_id'] = person_rr['person_id'].astype(int)

print(f"Treated (railroad_interest=1): {coding['railroad_interest'].sum()}")
print(f"Confirmed controls:            {coding['confirmed_control'].sum()}")
print(f"Dropped (all NaN / uncoded):   {len(coding) - len(person_rr)}")
print(f"\nPersons in analysis: {len(person_rr)}")

## 2. Link owners → newspapers (issn + active years)

In [ ]:
oe = pd.read_csv(OE_PATH)

# Keep only rows with a known person and a valid issn
oe_persons = oe.dropna(subset=['person_id', 'issn']).copy()
oe_persons['person_id'] = oe_persons['person_id'].astype(int)

# Merge in railroad interest
oe_persons = oe_persons.merge(person_rr, on='person_id', how='inner')

# Expand year ranges: 'years' column is like "1871; 1873; 1876"
def parse_years(s):
    if pd.isna(s):
        return []
    return [int(y.strip()) for y in str(s).split(';') if y.strip().isdigit()]

rows = []
for _, row in oe_persons.iterrows():
    for yr in parse_years(row['years']):
        rows.append({'issn': row['issn'], 'year': yr,
                     'person_id': row['person_id'], 'railroad_interest': row['railroad_interest']})

person_paper_year = pd.DataFrame(rows)
print(f"Person-paper-year rows: {len(person_paper_year)}")
print(person_paper_year.head())

In [ ]:
# Aggregate to newspaper-year: 1 if ANY owner in that year had railroad interest
paper_year_rr = (
    person_paper_year
    .groupby(['issn', 'year'])['railroad_interest']
    .max()
    .reset_index()
    .rename(columns={'railroad_interest': 'railroad_interest'})
)

print(f"Newspaper-years with owner coding: {len(paper_year_rr)}")
print(paper_year_rr['railroad_interest'].value_counts())

## 3. Compute anti-labor intensity from article_sentiment

In [ ]:
conn = sqlite3.connect(DB_PATH)
sent = pd.read_sql("""
    SELECT issn, year, labor_sentiment
    FROM article_sentiment
    WHERE issn != '' AND labor_sentiment IS NOT NULL
""", conn)
conn.close()

# Count labor articles by sentiment class per newspaper-year
counts = (
    sent.groupby(['issn', 'year', 'labor_sentiment'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['anti_labor', 'pro_labor', 'neutral']:
    if col not in counts.columns:
        counts[col] = 0

counts['total_labor'] = counts['anti_labor'] + counts['pro_labor'] + counts['neutral']
counts['anti_labor_intensity'] = counts['anti_labor'] / counts['total_labor']

print(f"Newspaper-years with labor sentiment: {len(counts)}")
counts[['anti_labor', 'pro_labor', 'neutral', 'total_labor', 'anti_labor_intensity']].describe()

## 4. Merge and describe the analysis sample

In [ ]:
df = counts.merge(paper_year_rr, on=['issn', 'year'], how='inner')

print(f"Analysis sample: {len(df)} newspaper-years, {df['issn'].nunique()} newspapers")
print(f"railroad_interest=1: {df['railroad_interest'].sum()} obs | railroad_interest=0: {(df['railroad_interest']==0).sum()} obs")
print()
print(df.groupby('railroad_interest')[['anti_labor_intensity', 'total_labor']].describe().T)

## 5. Regression

In [ ]:
# Add state info for controls
oe_meta = oe[['issn', 'state']].drop_duplicates('issn')
df = df.merge(oe_meta, on='issn', how='left')
df['year_str'] = df['year'].astype(str)  # year fixed effects

df.head()

In [ ]:
# Model 1: bivariate
m1 = smf.ols('anti_labor_intensity ~ railroad_interest', data=df).fit(cov_type='HC3')
print(m1.summary())

In [ ]:
# Model 2: + year fixed effects
m2 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df).fit(cov_type='HC3')
print(m2.summary())

In [ ]:
# Model 3: + year + state fixed effects
m3 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + C(state)', data=df).fit(cov_type='HC3')
print(m3.summary())

In [ ]:
# Summary table
from statsmodels.iolib.summary2 import summary_col

print(summary_col(
    [m1, m2, m3],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) Year+State FE'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'Intercept']
))